In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd
import metpy.calc as mpcalc

#data classes
import xarray as xr

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarComparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"; spinup_hours="-16"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class,ERA5DataLoading_Class_gdex

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData_NSSL.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetNumElements():
    num_elements = np.arange(ModelData_NSSL.Ntime)[start_job:end_job].tolist()
    return num_elements
loop_elements = GetNumElements()

In [ ]:
########################
#DATA INFORMATION

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. X-Band Scanning ARM Cloud Radar (XSACRCFRQC), 2022-06-09 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by Y. Feng, A. Matthews, E. Schuman, K. Johnson, I. Lindenmaier, V. Castro and T. Wendler. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/2001296.

#Globus Download Link
# https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261781*__;Ly8v!!PvDODwlR4mBZyAb0!REKAOzHJNvJk50CY5Pjl135CV83BhArwtdyMDuBM-28KqreBug8Xb5Mc3MLgy_p9PUiWe2uVXq-EfUtLcGdH5w$

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. Ka-Band Scanning ARM Cloud Radar (KASACRCFRQC), 2022-06-08 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by I. Lindenmaier, K. Johnson, D. Nelson, A. Matthews, T. Wendler, V. Melo de Castro, M. Rocque and Y. Feng. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/1877338.

# https://armgov.svcs.arm.gov/capabilities/instruments/kasacr

#Globus Download Link
#https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261893*__;Ly8v!!PvDODwlR4mBZyAb0!UJrKWg_xaYuaaa_iaY91TCVMB_neSskXsDKHtPPCG7Ix6sEmiEvnngTUrYuV18LudZqxB3eHyTDuCHSZ3o3zrg$

In [ ]:
#LOADING RADAR CLASS
if int(ModelData_NSSL.spinup_hours) <= 0:
    if ModelData_NSSL.region == "TRACER":
        if ModelData_NSSL.case == "WET":
            dateString = '2022-06-30_2022-07-03'
        elif ModelData_NSSL.case == "DIURNAL":
            dateString = '2022-06-21_2022-06-24'
    elif ModelData_NSSL.region == "Hawaii":
        if ModelData_NSSL.case == "WET":
            dateString = "2021-12-05_2021-12-08"
        elif ModelData_NSSL.case == "TRADES":
            dateString = "2022-08-07_2022-08-10"
        
else:
    dateString = f"{ModelData_NSSL.simulationDates[0]}_{ModelData_NSSL.simulationDates[-1]}"

RadarData_MRMS = RadarData_MRMS_Class(ModelData_NSSL,
                                      fileDirectory=os.path.join(DirectoryManager.dataDirectory,
                                                                 f"Observation_Data/{ModelData_NSSL.region}/MRMS_RadarData",
                                                                 dateString,
                                                                 "MergedReflectivityQC_01.00"))

In [ ]:
##########################
#DATA LOADING FUNCTIONS

In [ ]:
z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
zlevels = np.loadtxt(z_levels_filePath)/1e3
zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])

In [ ]:
#Converting timeStrings
def ConvertTimeStringtoDateTime(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    """
    return datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
def ConvertTimeStringtoTimeTitle(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    Formatted for use as a plot title.
    """
    # Parse the custom format to a datetime object
    dt = datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
    # Format it to 'YYYY-MM-DD HH:MM:SS'
    return dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
def ApplySmoothing(data, n_degree=8):
    # https://unidata.github.io/MetPy/latest/examples/calculations/Smoothing.html
    smooth = mpcalc.smooth_gaussian(data,n_degree)
    # smooth = mpcalc.smooth_n_point(data, 9)
    return smooth

def InterpolateModeltoERA5(model,observation):
    model_interp = model.interp(
        latitude=observation.latitude,
        longitude=observation.longitude,
        method="linear"
    )
    return model_interp

def FixLatLon_RadarData(radarData):        
    radarData = radarData.isel(latitude=slice(None, None, -1))

    radarData = radarData.assign_coords(
        longitude=((radarData.longitude + 180) % 360) - 180
    )
    return radarData

def ReturnLatLon_RadarData(radarData):
    # Fix latitude order
    radarData_fixed = radarData.isel(latitude=slice(None, None, -1))

    # Fix longitude convention
    radarData_fixed = radarData_fixed.assign_coords(
        longitude=radarData.longitude+360
    )

    return radarData_fixed

def InterpolateRadarData(radarData,modelData):
    radarData = FixLatLon_RadarData(radarData)
    
    radarData_interp = radarData.interp(
        latitude=modelData.latitude,
        longitude=modelData.longitude,
        method="linear"
    )
    radarData_interp = ReturnLatLon_RadarData(radarData_interp)
    return radarData_interp

In [ ]:
def GetData(t,z_km, option="two"):
    timeString = ModelData_NSSL.timeStrings[t]
    timeString_datetime = ConvertTimeStringtoDateTime(timeString)
    
    #Loading Model Radar
    z_idx = np.abs(zlevels_center - float(z_km)).argmin()
    # modelRadarData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)["refl10cm_1km"]
    # modelRadarData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)["refl10cm_1km"]
    modelRadarData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)["refl10cm"].isel(nVertLevels=z_idx)
    modelRadarData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)["refl10cm"].isel(nVertLevels=z_idx)
    model_z_km = np.round(zlevels_center[z_idx],2)
    modelRadarTimeTitle = ConvertTimeStringtoTimeTitle(timeString) + f" - {model_z_km} km"
    
    #Loading Observational Radar
    radarData, nearestFilePath = RadarData_MRMS.LoadClosestMRMSFile(target_time=timeString_datetime)
    radarData = InterpolateRadarData(radarData=radarData, modelData=modelRadarData_NSSL)
    radarTimeTitle = pd.to_datetime(radarData['time'].data[0]).strftime("%Y-%m-%d %H:%M:%S") + f" - {z_km} km"
    radarData=radarData.isel(time=0)

    #Getting Model MSLP Data
    mslpData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)['mslp']/1e2
    mslpData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)['mslp']/1e2

    #Getting ERA5 MSLP Data
    mslp_ERA5_alltimes = ERA5DataLoading_Class_gdex.LoadERA5Data(timeString, ModelData_NSSL, DirectoryManager)
    mslp_ERA5 = ERA5DataLoading_Class_gdex.SelectNearestERA5Time(mslp_ERA5_alltimes, timeString)/1e2
    # mslp_ERA5_alltimes = ERA5DataLoading_Class.LoadERA5Data(DirectoryManager, ModelData_NSSL, variableName='msl',dataType='Surface')
    # mslp_ERA5 = ERA5DataLoading_Class.SelectNearestERA5Time(mslp_ERA5_alltimes, ModelData_NSSL.timeStrings[t])/1e2

    #smoothing data for comparison
    if option == "one":
        pass
    elif option == "two":
        mslpData_NSSL = ApplySmoothing(mslpData_NSSL)
        mslpData_TEMPO = ApplySmoothing(mslpData_TEMPO)
    elif option == "three":
        mslpData_NSSL = InterpolateModeltoERA5(mslpData_NSSL,mslp_ERA5)
        mslpData_TEMPO = InterpolateModeltoERA5(mslpData_TEMPO,mslp_ERA5)

    return (
    modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
    radarData,radarTimeTitle, 
    timeString,
        
    mslpData_NSSL,mslpData_TEMPO,mslp_ERA5
    )

In [ ]:
##########################
#PLOTTING FUNCTIONS

In [ ]:
terrainData = ModelData_NSSL.staticData['ter']
def PlotTerrain(axis,zorder=-10):
    terrainLevels = np.linspace(
        0,float(terrainData.max()),20)
    cs = axis.contour(terrainData.longitude, terrainData.latitude, terrainData, levels=terrainLevels, cmap="terrain", alpha=0.2, zorder=zorder)

In [ ]:
def MakePlot(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
             radarData,radarTimeTitle,
             mslpData_NSSL,mslpData_TEMPO,mslp_ERA5):
    
    fig, axes = RadarPlotting_Class.CreateMapAxes(nrows=1,ncols=3,
                                                  figsize=(16,8))
    
    #Plotting ModelRadar
    #nssl
    axis = axes[0,0]
    lat = modelRadarData_NSSL['latitude']
    lon = modelRadarData_NSSL['longitude']
    contourPlot = RadarPlotting_Class.PlotReflectivity(axis, lat,lon,modelRadarData_NSSL,dataName="NSSL",timeTitle=modelRadarTimeTitle)

    #Adding MSLP Contours
    lat = mslpData_NSSL['latitude']
    lon = mslpData_NSSL['longitude']
    cs1 = axis.contour(lon, lat, mslpData_NSSL, 
                       colors='black', levels=10, linewidths=1.0, alpha=0.35, zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #tempo
    axis = axes[0,2]
    lat = modelRadarData_TEMPO['latitude']
    lon = modelRadarData_TEMPO['longitude']
    RadarPlotting_Class.PlotReflectivity(axis, lat,lon,modelRadarData_TEMPO,dataName="TEMPO",timeTitle=modelRadarTimeTitle)

    #Adding MSLP Contours
    lat = mslpData_TEMPO['latitude']
    lon = mslpData_TEMPO['longitude']
    cs1 = axis.contour(lon, lat, mslpData_TEMPO, 
                       colors='black', levels=10, alpha=0.35, linewidths=1.0,zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #Plotting Observational Radar
    #mrms data
    axis = axes[0,1]
    lat = radarData['latitude'].data
    lon = radarData['longitude'].data-360
    
    RadarPlotting_Class.PlotReflectivity(axis, lat,lon,radarData,dataName="MRMS",timeTitle=radarTimeTitle)

    #Adding MSLP Contours
    cs1 = axis.contour(mslp_ERA5.longitude, mslp_ERA5.latitude, mslp_ERA5, 
                       colors='black', levels=10, alpha=0.45, linewidths=1.0,zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #Adding Colorbar
    colorBar = RadarPlotting_Class.AddSharedColorbar(fig, contourPlot)

    #Adding Terrain
    for axis in fig.get_axes():
        PlotTerrain(axis, zorder=-10)
    return fig

In [ ]:
def AddMaskContour(fig, RadarDataMask):
    axes = fig.get_axes()
    for axis in axes:
        axis.contour(RadarDataMask.longitude,RadarDataMask.latitude,RadarDataMask*1,colors='red',linewidths=0.2,zorder=100)

In [ ]:
def GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory):
    outputSubDirectory = f"{ModelData_1.region}_{ModelData_1.case}_{ModelData_1.spinup_hours}hrs"

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        "RadarComparison_Animation",
        outputSubDirectory)
    os.makedirs(outputFilePath, exist_ok=True)
    return outputFilePath
    
def SaveFigure(fig, ModelData_1,ModelData_2, timeString):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"RadarComparison_{ModelData_1.mpType}vsMRMSvs{ModelData_2.mpType}_{timeString}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
##########################
#PLOTTING

In [ ]:
#Loading RadarDataMask
RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)
z_km=4 if ModelData_NSSL.region == "PRECIP" else 1

In [ ]:
for t in tqdm(loop_elements, desc="Processing"):
    [modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
     radarData,radarTimeTitle,
     timeString,
     mslpData_NSSL,mslpData_TEMPO,mslp_ERA5]=GetData(t,z_km)
    
    fig = MakePlot(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
                   radarData,radarTimeTitle,
                   mslpData_NSSL,mslpData_TEMPO,mslp_ERA5)

    #Adding RadarDataMask
    AddMaskContour(fig, RadarDataMask)
    
    SaveFigure(fig, ModelData_NSSL,ModelData_TEMPO, timeString)

In [ ]:
#################
#MAKING ANIMATION
ANIMATE=False #keep false when running with bash code
ANIMATE=True

In [ ]:
if ANIMATE==True:    
    #Importing AnimationPlotting_Class
    sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
    from CLASSES_PlottingModelData import AnimationPlotting_Class

In [ ]:
def GetVariableInputFiles(ModelData_1,ModelData_2, outputPlottingDirectory):
    filePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    fileName = f"*.png"
    filePattern = DirectoryManager.GetOutputFile(outputPlottingDirectory, filePath, fileName)
    print(filePattern)
    filePaths = DirectoryManager.GetSortedFileListByTimestamp(filePattern)
    return filePaths

def GetPlottingFileName(ModelData_1,ModelData_2, outputPlottingDirectory, filePaths, extension="mp4"):
    filePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)

    splits = os.path.basename(filePaths[0]).split('_')
    plottingFileName = f"{splits[0]}_{splits[1]}.{extension}"
    
    plottingFilePath = DirectoryManager.GetOutputFile(outputPlottingDirectory, filePath, plottingFileName)
    return plottingFilePath

In [ ]:
# PNGtoMP4 VERSION
if ANIMATE==True:    
    # Setting up output file
    print("getting file information")
    imageFiles = GetVariableInputFiles(ModelData_NSSL,ModelData_TEMPO, outputPlottingDirectory)
    plottingFilePath = GetPlottingFileName(ModelData_NSSL,ModelData_TEMPO, outputPlottingDirectory, imageFiles, extension="mp4")

     # running animation
    fps = AnimationPlotting_Class.CalculateFPS(num_frames=ModelData_NSSL.Ntime, time_interval_minutes=15, desired_duration_min=1)
    AnimationPlotting_Class.PNGsToMP4(imageFiles, plottingFilePath, fps=fps)

In [ ]:
####################################
#PLOTTING ALL SIMULATIONS
plotting = False #keep false when job array is running
plotting = True

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
                      timeString,
                      extension="png"):

    inputFilePath = os.path.join(
        outputPlottingDirectory,
        "RadarComparison_Animation")
    
    radarDataName = "MRMS" if region != "PRECIP" else "PRECIP"
    fileName=f"RadarComparison_NSSLvs{radarDataName}vsTEMPO_{timeString}" 

    
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(inputFilePath, inputSubDirectory)
    # --- File path ---
    inputFilePath = os.path.join(
        load_dir,
        f"{fileName}.{extension}"
    )
    return inputFilePath

def GetFilePaths(timeStringDictionary):

    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        timeString = timeStringDictionary[region][case][spinup_hours]
        filePaths.append(GetFigureFilePath(region,case,spinup_hours,
                                           timeString))
    return filePaths

In [ ]:
if plotting:
    sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
    from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:
if plotting:
    timeStringDictionary = {
        "TRACER": {
            "WET": {
                0: f"2022-07-01_18.00.00",
            },
            "DIURNAL": {
                -5: f"2022-06-22_21.00.00",
            },
        },
        "PRECIP": {
            "WET": {
                12: f"2022-06-07_03.00.00",
            },
            "DIURNAL": {
                12: f"2022-07-16_05.00.00",
            },
        },
        "Hawaii": {
            "WET": {
                12: f"2021-12-06_23.00.00",
            },
            "TRADES": {
                24: f"2022-08-08_01.00.00",
            },
        },
    }
# /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING/DataAnalysis/Observation_Data/RadarComparison/RadarComparison_Animation

In [ ]:
if plotting:
    filePaths = GetFilePaths(timeStringDictionary)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 4),
                                                     wspace=0.02,hspace=0.02,
                                                     dpi=600)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig, saveDirectory=os.path.join(outputPlottingDirectory,"RadarComparison_Animation"),fileName="RadarComparison_Animation")